> **対応するブログ記事**: [#4 商用利用OKのsage-proteomicsでDIA-MSデータを解析する](../blog/article-04-sage.md)
>
> このNotebookはブログ記事 #4 のコードをセルごとに実行できるインタラクティブ版です。処理の背景や詳しい解説はブログ記事を参照してください。

# Step 4: sage DIA解析 → タンパク質マトリクス構築

DIA-MS の mzML ファイルからタンパク質定量マトリクスを構築します。
1. **mzML 検査** — pymzml でスペクトル数・RT 範囲を確認
2. **sage 実行** — library-free DIA 解析で PSM + LFQ を計算
3. **マトリクス構築** — lfq.tsv を Gene Symbol × サンプルに集約

## Part 1: mzML ファイルの検査

In [ ]:
# ファイル操作・パターンマッチ・正規表現・時間計測・外部プロセス実行・システム情報のための標準ライブラリ
import os, glob, re, time, subprocess, sys
# データフレーム操作のための pandas ライブラリ
import pandas as pd
# mzML 形式の質量分析データを読み込むためのライブラリ
import pymzml

In [ ]:
# mzMLファイルのパスを格納するディレクトリ（DIA-MSの生データが入っている）
MZML_DIR    = "../data/raw/raw_mzML"
# ヒトプロテオームのFASTA配列ファイル（sageがペプチド照合に使う参照データベース）
FASTA_PATH  = "../data/raw/human_proteome.fasta"
# 解析結果全般を保存するディレクトリ
RESULTS_DIR = "../results"
# sage の設定ファイル（酵素切断ルール・質量許容差・修飾などを定義）
SAGE_CONFIG = "../scripts/sage_config.json"
# sage の出力先ディレクトリ（lfq.tsv や results.sage.tsv が生成される）
SAGE_OUT    = os.path.join(RESULTS_DIR, "sage_output")
# 中間テーブル（mzML のインベントリCSV等）を保存するディレクトリ
TABLES_DIR  = os.path.join(RESULTS_DIR, "tables")

# TABLES_DIR が存在しなければ作成（exist_ok=True で既存でもエラーにならない）
os.makedirs(TABLES_DIR, exist_ok=True)
# SAGE_OUT が存在しなければ作成
os.makedirs(SAGE_OUT, exist_ok=True)

In [ ]:
def inspect_mzml(path):
    """mzML を走査して MS1/MS2 数と RT 範囲を返す。"""
    # pymzml でmzMLファイルをパースするリーダーオブジェクトを生成
    reader = pymzml.run.Reader(path)
    # MS1（フルスキャン）とMS2（フラグメントスキャン）のスペクトル数カウンタを初期化
    ms1, ms2 = 0, 0
    # 保持時間（RT）の最小値・最大値を追跡する変数（初期値は極値に設定）
    rt_min, rt_max = float("inf"), float("-inf")

    # 全スペクトルを1つずつ走査するループ
    for spec in reader:
        # 現在のスペクトルの保持時間を分単位で取得
        rt = spec.scan_time_in_minutes()
        # RTがNoneでなければ最小・最大値を更新
        if rt is not None:
            rt_min, rt_max = min(rt_min, rt), max(rt_max, rt)
        # MS1スペクトル（プリカーサーイオンの全体像）ならカウントを加算
        if spec.ms_level == 1:
            ms1 += 1
        # MS2スペクトル（フラグメントイオン、ペプチド同定に使用）ならカウントを加算
        elif spec.ms_level == 2:
            ms2 += 1

    # ファイル名・サイズ・スペクトル数・RT範囲を辞書として返す
    return {
        "file": os.path.basename(path),            # ファイル名のみ（ディレクトリを除去）
        "size_MB": round(os.path.getsize(path) / 1024**2, 1),  # ファイルサイズをMB単位で計算
        "ms1": ms1,                                 # MS1スペクトル総数
        "ms2": ms2,                                 # MS2スペクトル総数
        "rt_min": round(rt_min, 2),                 # 保持時間の最小値（分）
        "rt_max": round(rt_max, 2),                 # 保持時間の最大値（分）
    }

In [ ]:
# MZML_DIR 内の全 .mzML ファイルをソートして取得（CRC01-N, CRC01-T, ... の順）
mzml_files = sorted(glob.glob(os.path.join(MZML_DIR, "*.mzML")))
# 見つかったファイル数を表示して、データの欠損がないか確認
print(f"{len(mzml_files)} mzML files found")

# 各mzMLファイルに対して inspect_mzml を実行し、検査結果のリストを作成
records = [inspect_mzml(f) for f in mzml_files]
# 検査結果のリストをDataFrameに変換（ファイルごとの情報を表形式で確認できる）
df_inv = pd.DataFrame(records)
# インベントリCSVとして保存（後で品質管理や再確認に使える）
df_inv.to_csv(os.path.join(TABLES_DIR, "mzml_inventory.csv"), index=False)
# DataFrameを表示してスペクトル数やRT範囲に異常がないか目視確認
df_inv

## Part 2: sage 実行

論文では DIA-NN v1.8.1 を使用していますが、有料ライセンスが必要なため
MIT ライセンスの **sage-proteomics** で代替します（Apple Silicon で約4.5分）。

In [ ]:
# sage コマンドライン引数を組み立てる（FASTA、出力先、テレメトリ無効化、設定ファイル + mzMLファイル群）
cmd = [
    "sage",                         # sage-proteomics の実行バイナリ
    "--fasta", FASTA_PATH,          # ペプチド照合に使うヒトプロテオームFASTAファイル
    "--output_directory", SAGE_OUT,  # 結果ファイル（lfq.tsv等）の出力先ディレクトリ
    "--disable-telemetry-i-dont-want-to-improve-sage",  # テレメトリ送信を無効化
    SAGE_CONFIG,                    # 検索パラメータを定義したJSON設定ファイル
] + mzml_files  # 解析対象の全mzMLファイルパスをリストに追加

# 解析開始メッセージを表示（ファイル数を確認）
print(f"Running sage on {len(mzml_files)} files...")
# 処理時間計測のために開始時刻を記録
t0 = time.time()

# sage をサブプロセスとして起動（標準出力と標準エラーを統合してパイプで受け取る）
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
# sage の出力をリアルタイムで1行ずつ表示（進捗確認のため）
for line in proc.stdout:
    sys.stdout.write(line)
# プロセスの終了を待機し、終了コードを取得
proc.wait()

# 経過時間と終了コードを表示（exit code 0 なら正常終了）
print(f"\nDone in {(time.time() - t0) / 60:.1f} min (exit code {proc.returncode})")

## Part 3: タンパク質マトリクス構築

sage の `lfq.tsv`（ペプチドレベル LFQ）を FASTA の GN= フィールドで
Gene Symbol にマップし、Gene × サンプルの定量マトリクスに集約します。

In [ ]:
def parse_fasta_gene_map(fasta_path):
    """UniProt FASTA ヘッダから accession -> Gene Symbol の辞書を作る。

    ヘッダ例: >sp|P04637|P53_HUMAN ... GN=TP53 ...
    GN= がなければ Entry Name の先頭語 (例: P53) を使う。
    """
    # アクセッション番号をキー、Gene Symbolを値とする辞書
    id_to_gene = {}
    # FASTAヘッダ内の "GN=遺伝子名" を抽出する正規表現パターン
    gn_re = re.compile(r"\bGN=(\S+)")
    # FASTAファイルを行ごとに読み込む
    with open(fasta_path) as f:
        for line in f:
            # ">" で始まらない行は配列データなのでスキップ
            if not line.startswith(">"):
                continue
            # ヘッダ行を "|" で最大3パートに分割（例: sp, P04637, P53_HUMAN ...）
            parts = line[1:].split("|", 2)
            if len(parts) >= 3:
                # 2番目の要素がUniProtアクセッション番号（例: P04637）
                acc = parts[1]
                # 3番目の要素の最初の単語がエントリー名（例: P53_HUMAN）
                entry_name = parts[2].split()[0]
            else:
                # パイプ区切りでない場合は最初の単語をアクセッション番号として使用
                acc = line[1:].split()[0]
                entry_name = acc
            # ヘッダ行から GN=（Gene Name）フィールドを正規表現で検索
            m = gn_re.search(line)
            # GN= が見つかればその値を、なければエントリー名の "_" 前部分を遺伝子名として使用
            id_to_gene[acc] = m.group(1) if m else entry_name.split("_")[0]
    # アクセッション → Gene Symbol の辞書を返す
    return id_to_gene

In [ ]:
# ペプチド FDR 閾値（論文と同じ 1%、偽陽性率を1%以下に制御）
Q_THRESHOLD = 0.01

# sage が出力した LFQ（Label-Free Quantification）結果のTSVファイルパス
LFQ_TSV = os.path.join(SAGE_OUT, "lfq.tsv")
# LFQファイルをタブ区切りで読み込み、ペプチドレベルの定量データを取得
df = pd.read_csv(LFQ_TSV, sep="\t")
# ペプチド数とカラム数を表示して、期待通りのデータが読めているか確認
print(f"lfq.tsv: {len(df)} peptides, {len(df.columns)} columns")

# q_value（FDR推定値）が閾値未満のペプチドのみを残す（偽陽性を除去）
df = df[df["q_value"] < Q_THRESHOLD].copy()
# フィルタ後に残ったペプチド数を表示
print(f"After q < {Q_THRESHOLD}: {len(df)} peptides")

# メタデータ列の名前セット（ペプチド配列・電荷・タンパク質名・統計量など）
meta_cols = {"peptide", "charge", "proteins", "q_value", "score", "spectral_angle"}
# メタ列以外がサンプルごとのLFQ強度値列なので、それらを抽出
sample_cols = [c for c in df.columns if c not in meta_cols]

# カラム名から ".mzML" サフィックスを除去して、サンプル名を簡潔にする
rename = {c: c.replace(".mzML", "") for c in sample_cols}
# DataFrameのカラム名をリネーム
df = df.rename(columns=rename)
# リネーム後のサンプル名リストを更新
sample_cols = list(rename.values())
# サンプル数を表示（16患者 × 2条件 = 32が期待値）
print(f"Samples: {len(sample_cols)}")

In [ ]:
# FASTAファイルを解析して、UniProtアクセッション番号 → Gene Symbol の対応辞書を構築
id_to_gene = parse_fasta_gene_map(FASTA_PATH)
# FASTAに含まれるエントリ数を表示（ヒトプロテオームなら約8万件が目安）
print(f"FASTA entries: {len(id_to_gene)}")

# proteins 列 "sp|ACC|NAME;..." から最初のタンパク質のアクセッション番号を抽出
# ";" で複数タンパク質が含まれる場合は代表（先頭）のみ使用
df["acc"] = df["proteins"].str.split(";").str[0].str.split("|").str[1]
# アクセッション番号をGene Symbolにマッピング（マッピングできなければacc自体を使用）
df["gene"] = df["acc"].map(id_to_gene).fillna(df["acc"])

# Gene Symbol ごとにペプチドのLFQ強度値を合計して、タンパク質レベルの定量マトリクスを構築
# min_count=1 により、全てNaNの場合はNaNを維持（0にしない）
matrix = df.groupby("gene")[sample_cols].sum(min_count=1)
# 強度値が0のセルをNAに変換（未検出を欠損として扱う）し、全行NAの行を除去
matrix = matrix.replace(0, pd.NA).dropna(how="all")
# インデックス名を "Protein" に設定（下流の可視化や差次解析で使いやすくするため）
matrix.index.name = "Protein"
# マトリクスのサイズを表示（タンパク質数 × サンプル数）
print(f"Protein matrix: {matrix.shape[0]} proteins x {matrix.shape[1]} samples")
# 先頭5行を表示して、マトリクスの構造と値を目視確認
matrix.head()

In [ ]:
# タンパク質定量マトリクスの保存先パスを設定
out_csv = os.path.join(RESULTS_DIR, "protein_matrix_from_sage.csv")
# Gene Symbol × サンプルのマトリクスをCSVファイルとして保存（後続ステップで使用）
matrix.to_csv(out_csv)
# 保存先パスを表示して確認
print(f"Saved: {out_csv}")